## 데이터 수집 자동화

### 다나와 상품정보 스크래핑

#### 자동 로그인

- 패스워드 숨기기 

##### 1. setx 명령어로 시스템에 저장방법
- powershell

```powershell
> setx DANAWA_ID "아이디"
성공: 지정한 값을 저장했습니다.

> setx DANAWA_PASSWORD "비밀번호"

성공: 지정한 값을 저장했습니다.
```

- sysdm.cpl로 시스템 정보에서 환경변수 user 리스트에서 오픈

##### 2. keyring 으로 윈도우 자격증명 저장소

- setx 와 달리 아예 오픈 안됨
- 패키지 설치

In [6]:
!pip install keyring

Defaulting to user installation because normal site-packages is not writeable

   --------------------------------- ------ 5/6 [keyring]
   ---------------------------------------- 6/6 [keyring]



In [7]:
import keyring
from getpass import getpass

In [14]:
username = input('다나와 아이디')
password = getpass('다나와 비밀번호') 

# 아이디 저장
keyring.set_password('danawa.com', 'user_id', username)
# 패스워드
keyring.set_password('danawa.com', username, password)

print('아이디/패스워드 저장 완료')

아이디/패스워드 저장 완료


In [15]:
user_id = keyring.get_password('danawa.com', 'user_id')
user_id

'personar'

In [19]:
password = keyring.get_password('danawa.com', user_id)
password is not None

True

- 패스워드 오픈 안되고

In [1]:
import pandas as pd
import os

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

In [2]:
driver = webdriver.Chrome()
wait = WebDriverWait(driver, 10)
driver.get('https://www.danawa.com/')

wait.until(
    EC.presence_of_element_located((By.ID, 'danawa_footer'))
)

<selenium.webdriver.remote.webelement.WebElement (session="2decfbd75906d4e41b21654a8295afc6", element="f.B432447360C3A4B88DAACFEE8A3D3845.d.C0D14E81F52E53C4B9A2814802E0BBB0.e.81")>

In [22]:
# 로그인 링크 확인후 클릭
driver.find_element(By.LINK_TEXT, '로그인').click()

In [23]:
# 로그인하기
id_input = driver.find_element(By.CSS_SELECTOR, 'input[type="text"]')
id_input.send_keys(user_id)

pass_input = driver.find_element(By.CSS_SELECTOR, 'input[type="password"]')
pass_input.send_keys(password)

In [25]:
login_btn = driver.find_element(By.CLASS_NAME, 'btn_login')
login_btn.click()

#### 동적 페이지 스크래핑

- https://prod.danawa.com/list/?cate=11255834&15main_11_02

In [3]:
driver.get('https://prod.danawa.com/list/?cate=11255834&15main_11_02')

wait.until(
    EC.presence_of_element_located((By.TAG_NAME, 'footer'))
)

<selenium.webdriver.remote.webelement.WebElement (session="2decfbd75906d4e41b21654a8295afc6", element="f.B432447360C3A4B88DAACFEE8A3D3845.d.B86528C7DEFD49187484867BBD66A46D.e.6603")>

- select 태그 페이지 내에 1개만 존재

In [4]:
options = driver.find_elements(By.TAG_NAME, 'option')

for option in options:
    if option.get_attribute('value') == '90':
        option.click()

wait.until(
    EC.presence_of_element_located((By.TAG_NAME, 'footer'))
)

<selenium.webdriver.remote.webelement.WebElement (session="2decfbd75906d4e41b21654a8295afc6", element="f.B432447360C3A4B88DAACFEE8A3D3845.d.B86528C7DEFD49187484867BBD66A46D.e.6603")>

In [5]:
products = driver.find_elements(By.CSS_SELECTOR, 'div[data-testid="ProductListItem"]')
len(products)

90

In [6]:
product = products[0]
product.text

'1\n영웅컴퓨터 영웅 울트라게이밍 9800XS R7 9800X3D RTX5080\n9800X3D / RTX 5080 / (AMD) B650 / OS미포함 / 900W / AMD / 라이젠 9000시리즈 / 라이젠7 / 그래니트 릿지 / DDR5 / 32GB / M.2 / 1TB / NVIDIA / 그래픽 메모리: 16GB / 1Gbps 유선 / HDMI / DP포트 / 파워서플라이 / 미들타워 / 어항형 / 수랭쿨러 / LED쿨러 / 용도: 게임용\n혜택 최저가\n4,379,770원\n[G마켓] 삼성카드\n/\n무이자 최대 24개월\n기획전\n오늘 사는 PC가 가장 저렴한 PC!\n24.12. 등록\n브랜드로그\n4.8\n리뷰수\n(11)\n관심\n대량구매\n32GB, M.2 1TB\n4,529,810\n원\n5몰\nVS상품비교에 추가\nNew 새로워진 상품비교 기능을 이용해보세요.'

In [7]:
## 1번 인덱스 값에서 제품명, 제품스펙, 2번 인덱스 값에서 가격 
product.find_elements(By.CSS_SELECTOR, '.dnw-product-list-item > div')


[<selenium.webdriver.remote.webelement.WebElement (session="2decfbd75906d4e41b21654a8295afc6", element="f.B432447360C3A4B88DAACFEE8A3D3845.d.B86528C7DEFD49187484867BBD66A46D.e.6892")>,
 <selenium.webdriver.remote.webelement.WebElement (session="2decfbd75906d4e41b21654a8295afc6", element="f.B432447360C3A4B88DAACFEE8A3D3845.d.B86528C7DEFD49187484867BBD66A46D.e.6893")>,
 <selenium.webdriver.remote.webelement.WebElement (session="2decfbd75906d4e41b21654a8295afc6", element="f.B432447360C3A4B88DAACFEE8A3D3845.d.B86528C7DEFD49187484867BBD66A46D.e.6894")>]

In [8]:
computer_info = product.find_elements(By.CSS_SELECTOR, '.dnw-product-list-item > div')[1]
computer_info.text

'1\n영웅컴퓨터 영웅 울트라게이밍 9800XS R7 9800X3D RTX5080\n9800X3D / RTX 5080 / (AMD) B650 / OS미포함 / 900W / AMD / 라이젠 9000시리즈 / 라이젠7 / 그래니트 릿지 / DDR5 / 32GB / M.2 / 1TB / NVIDIA / 그래픽 메모리: 16GB / 1Gbps 유선 / HDMI / DP포트 / 파워서플라이 / 미들타워 / 어항형 / 수랭쿨러 / LED쿨러 / 용도: 게임용\n혜택 최저가\n4,379,770원\n[G마켓] 삼성카드\n/\n무이자 최대 24개월\n기획전\n오늘 사는 PC가 가장 저렴한 PC!\n24.12. 등록\n브랜드로그\n4.8\n리뷰수\n(11)\n관심\n대량구매'

In [9]:
# 컴퓨터 이름 추출!!
computer_info.find_element(By.CSS_SELECTOR, 'div a').text

'영웅컴퓨터 영웅 울트라게이밍 9800XS R7 9800X3D RTX5080'

In [10]:
# 컴퓨터 사양
specs = computer_info.find_elements(By.CSS_SELECTOR, 'div[data-testid="ProductListSpecs"]')
' / '.join(spec.text for spec in specs)

'9800X3D / RTX 5080 / (AMD) B650 / OS미포함 / 900W / AMD / 라이젠 9000시리즈 / 라이젠7 / 그래니트 릿지 / DDR5 / 32GB / M.2 / 1TB / NVIDIA / 그래픽 메모리: 16GB / 1Gbps 유선 / HDMI / DP포트 / 파워서플라이 / 미들타워 / 어항형 / 수랭쿨러 / LED쿨러 / 용도: 게임용'

In [11]:
price_info = product.find_elements(By.CSS_SELECTOR, '.dnw-product-list-item > div')[2]
price_info.text

'32GB, M.2 1TB\n4,529,810\n원\n5몰\nVS상품비교에 추가\nNew 새로워진 상품비교 기능을 이용해보세요.'

In [15]:
price_info.find_element(By.CSS_SELECTOR,'li').text

'32GB, M.2 1TB\n4,529,810\n원\n5몰\nVS상품비교에 추가\nNew 새로워진 상품비교 기능을 이용해보세요.'

In [16]:
# 인덱스로 데이터 가져오면서 필요값 필터링
price_info.find_elements(By.CSS_SELECTOR,'li > div span')[0].text

'32GB, M.2 1TB'

In [17]:
spans = price_info.find_elements(By.CSS_SELECTOR,'li > div span')

for span in spans:
    if ('위' in span.text): continue
    if ('원' in span.text): break
    print(span.text)

32GB, M.2 1TB
4,529,810


In [19]:
# 다나와 게시판 게이밍PC 페이지로 이동
driver.get('https://prod.danawa.com/list/?cate=11255834&15main_11_02')

wait.until(
    # 모든 요소가 화면에 완전표시될때까지 대기
    EC.visibility_of_all_elements_located((By.TAG_NAME, 'footer'))
)

# 게시물 개수 30개에서 90개로 늘리기
options = driver.find_elements(By.TAG_NAME, 'option')

for option in options:
    if option.get_attribute('value') == '90':
        option.click()

# 화면에 90개 데이터를 다 조회할때까지 대기
products = wait.until(
    EC.presence_of_all_elements_located((By.CSS_SELECTOR, 'div[data-testid="ProductListItem"]'))
)

# list를 담을 수 있는 배열 생성
danawa_prices = []

# 총 게시물 읽어오기
# len(products)
for index, product in enumerate(products):
    # print(index, product.text)
    items = product.find_elements(By.CSS_SELECTOR, '.dnw-product-list-item > div')
    computer_info = items[1]  # 컴퓨터 명, 사양 정보 들어있는 영역
    price_info = items[2]   # 가격정보 들어있는 영역

    # print(computer_info.text)
    ## Pandas DF에 컬럼으로 사용할 이름들 사용
    ## index, computer_name, computer_spec
    computer_name = computer_info.find_element(By.CSS_SELECTOR, 'div a').text
    specs = computer_info.find_elements(By.CSS_SELECTOR, 'div[data-testid="ProductListSpecs"]')
    computer_spec = ' / '.join(spec.text for spec in specs)

    # 가격
    spans = price_info.find_elements(By.CSS_SELECTOR, 'li > div span') 
    price_spec = ''
    price = ''

    for span in spans:
        if ('위' in span.text): continue  # 순위 제외
        if ('원' in span.text): break    # 첫번째 금액만 받고 종료

        if ('B' in span.text):
            price_spec = span.text
        else:
            price = span.text

    danawa_prices.append({
        'index': index,
        'computer_name': computer_name,
        'computer_spec': computer_spec,
        'price_spec' : price_spec,
        'price' : price
    })       
    

ElementNotInteractableException: Message: element not interactable: Element is not currently visible and may not be manipulated
  (Session info: chrome=154.0.8037.98); For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#elementnotinteractableexception
Stacktrace:
	chromedriver!GetHandleVerifier [0x7ff6aa1f9d55+5a55]
	chromedriver!(No symbol) [0x7ff6aa11f540]
	chromedriver!(No symbol) [0x7ff6a9f445fd]
	chromedriver!(No symbol) [0x7ff6a9f4c6e1]
	chromedriver!(No symbol) [0x7ff6a9f4f625]
	chromedriver!(No symbol) [0x7ff6a9f4f6ef]
	chromedriver!(No symbol) [0x7ff6a9fa4533]
	chromedriver!(No symbol) [0x7ff6a9f948d9]
	chromedriver!(No symbol) [0x7ff6a9fc90ea]
	chromedriver!(No symbol) [0x7ff6a9f930a6]
	chromedriver!(No symbol) [0x7ff6a9fefcd9]
	chromedriver!(No symbol) [0x7ff6a9f91a12]
	chromedriver!(No symbol) [0x7ff6a9f92823]
	chromedriver!GetHandleVerifier [0x7ff6aa6104b1+41c1b1]
	chromedriver!GetHandleVerifier [0x7ff6aa63

In [ ]:
len( danawa_prices)

In [ ]:
df_danawa = pd.DataFrame(danawa_prices)

In [ ]:
df_danawa.tail()

페이지 확인

In [21]:
#
buttons = driver.find_elements(By.CSS_SELECTOR, 'nav > nav div button')
len(buttons)

0

In [23]:
button_nextpage = buttons[9]
button_nextpage.click()

IndexError: list index out of range

In [24]:
# 다 등장했을 때 10페이지에 있는지 확인
for button in buttons:
    button.click()

    products = wait.until(
        EC.presence_of_all_elements_located(
            (By.CSS_SELECTOR, 'div[data-testid="ProductListItem"]')
        )
    )